<center><h1><b>LET'S FIND $D^0$ DECAYS</b></h1></center>
For computation problems, we now try to analyze the data (import + search for decay pairs) slicing them in more parts. So we slice each file in more parts to be analyzed separately.

In [1]:
import numpy as np
import pandas as pd
import awkward as ak
import uproot
import matplotlib.pyplot as plt
from IPython.display import Image, display

import gc
import itertools

### PATHS OF THE DATA
These are all the paths to the data.

```python
"""
Paths:
Chunk0950                    /Chunk0950/001_007/AO2Dtree.root
Chunk1000                    [ /Chunk1000/001_005/AO2Dtree.root, /Chunk1000/006_009/AO2Dtree.root ]
Chunk1010                    [ /Chunk1010/001_005/AO2Dtree.root, /Chunk1010/006_010/AO2Dtree.root ]
Chunk1020                    /Chunk1020/001_007/AO2Dtree.root
Chunk1030                    [ /Chunk1030/001_005/AO2Dtree.root, /Chunk1030/006_011/AO2Dtree.root ]
Chunk1040                    /Chunk1040/001_007/AO2Dtree.root
Chunk1050                    [ /Chunk1050/001_005/AO2Dtree.root, /Chunk1050/006_010/AO2Dtree.root ]
Chunk1100                    [ /Chunk1100/001_005/AO2Dtree.root, /Chunk1100/006_009/AO2Dtree.root ]
Chunk1110                    /Chunk1110/001_005/AO2Dtree.root
Chunk1120                    [ /Chunk1120/001_005/AO2Dtree.root, /Chunk1120/006_011/AO2Dtree.root ]
Chunk1130                    /Chunk1130/001_008/AO2Dtree.root
Chunk1140                    [ /Chunk1140/001_006/AO2Dtree.root, /Chunk1120/007_011/AO2Dtree.root ]
Chunk1150                    /Chunk1150/001_007/AO2Dtree.root
Chunk1200                    [ /Chunk1200/001_006/AO2Dtree.root, /Chunk1120/007_011/AO2Dtree.root ]
Chunk1210                    /Chunk1210/001_008/AO2Dtree.root
Chunk1220                    /Chunk1220/001_008/AO2Dtree.root
Chunk1300                    [ /Chunk1300/001_006/AO2Dtree.root, /Chunk1300/003/AO2Dtree.root, /Chunk1300/007_010/AO2Dtree.root ]
"""
```

### SELECTING THE CHUNK

In [2]:
base_path = "../../mnt/SingleTrackTrees/Data/alice_data_2023_LHC23f_535087_apass4_1300_Thinner/Output/Run535087"
which_chunk = "Chunk1300"
which_number = "001_006"
which_file = "AO2Dtree.root"
path = base_path + "/".join(["/", which_chunk, which_number, which_file])
#path = "~/Downloads/AO2Dtree.root"    # for when we run in local
file = uproot.open(path)

In [3]:
# NSigmaTPC:
sigma_limit = 3
cut1 = (
    "( (fNsigmaTPCpi > -3) & (fNsigmaTPCpi < 3) & (fCharge == 1) ) | "
    "( (fNsigmaTPCka > -3) & (fNsigmaTPCka < 3) & (fCharge == -1) ) | "
    "( (fPt > 0) & (fPt < 1) & (fNsigmaTPCka > -15) & (fNsigmaTPCka < 0) & (fCharge == 1) )"
)

# NsigmaTOF:
sigma_limit = 3
cut2 = (
    "( (fNsigmaTOFpi > -3) & (fNsigmaTOFpi < 3) & (fCharge == 1) ) | "
    "( (fNsigmaTOFka > -3) & (fNsigmaTOFka < 3) & (fCharge == -1) ) | "
    "( (fNsigmaTOFka > 998.5) | (fNsigmaTOFka < -998.5) | (fNsigmaTOFpi > 998.5) | (fNsigmaTOFpi < -998.5) ) | "
    "( (fPt > 0) & (fPt < 3) & (fNsigmaTOFka > -50) & (fNsigmaTOFka < 0) & (fCharge == 1) )"
)

# DCA_XY:
cut3 = "( (fDcaXY > 0.0002) | (fDcaXY < -0.0002) )"


# FINAL CUT EXPRESSION:
cut_expression = f"({cut1}) & ({cut2}) & ({cut3})"

In [4]:
def to_global_coord( row ):
    xy_in = np.array([row["fX"],row["fY"]])
    # angle = -row["fAlpha"]
    # rot = np.array([[np.cos(angle),np.sin(angle)], [-np.sin(angle), np.cos(angle)]]) # inverse matrix of the one from to_track_coord (-alfa)
    # Rewriting the above matrix using cosine/sin even/odd function properties: less multiplications
    angle = row["fAlpha"]
    rot = np.array([[np.cos(angle),-np.sin(angle)], [np.sin(angle), np.cos(angle)]])
    xy_out = rot.dot(xy_in)
    return( xy_out )

def secondary_vertex ( row1, row2 ):
    XY1, XY2 = [to_global_coord(row1), to_global_coord(row2)]
    x1 = XY1[0]
    y1 = XY1[1]
    x2 = XY2[0]
    y2 = XY2[1]
    
    px1 = row1["px"]
    px2 = row2["px"]
    py1 = row1["py"]
    py2 = row2["py"]
    pz1 = row1["pz"]
    pz2 = row2["pz"]
    
    m1 = py1/px1
    m2 = py2/px2
    q1 = y1 - m1*x1
    q2 = y2 - m2*x2
    x_SV = ( q2 - q1 )/( m1 - m2)
    y_SV = y1 + m1*(x_SV - x1)
   
    z1_track = pz1/px1 * x_SV + row1["fZ"]
    z2_track = pz2/px2 * x_SV + row2["fZ"]
    z_SV = (z1_track + z2_track)/2
    return ([x_SV, y_SV, z_SV])

In [5]:
# debug
names_dirs = file.keys(filter_classname="TDirectory")
subsets = np.array_split(range(0,len(names_dirs)),10)
subsets
len(subsets)

10

### DATA UPLOAD, FILTERING AND SEARCH FOR PAIRS

In [6]:
# Load T-Trees from directories
names_track_extr = file.keys(filter_name=r"*O2filtertrackextr")
names_track      = file.keys(filter_name=r"*O2filtertrack")
names_coll       = file.keys(filter_name=r"*O2collision_001")

# number of subsets to create: this is due to memory problems when doing the combinatorial
N_SPLITS = 10
subsets = np.array_split(range(0,len(names_coll )),N_SPLITS)

collision_offset = 0          # variable to correct the index of the collision (because it starts from 0 at each new TTree)

# Particle masses in GeV
m_K = 0.493677
m_pi = 0.139570
m_d0 = 1.86484

for J in range (len(subsets)):
    
    list_of_df = []               # add the dataframes in a list (we will concat them later)
    
    for i in subsets[J]:
    
        # Read collision tree
        df_coll = file[ names_coll[i] ].arrays(["fPosX", "fPosY", "fPosZ"], library="pd")   # I take the fPosZ column as a DataFrame
        # list_of_collision_df.append(df_coll)               # add the dataframe in a list (we will concat them later)
    
        # # Read track and trackextr using boolean mask for track and trackextr:
        df_trackextr = file[ names_track_extr[i] ].arrays(["fPt", "fEta", "fCharge", "fDcaXY",
             "fNsigmaTPCpi", "fNsigmaTPCka", "fNsigmaTPCpr", "fNsigmaTOFpi", "fNsigmaTOFka", "fNsigmaTOFpr"], library="pd" )
        df_track = file[ names_track[i] ].arrays(["fIndexCollisions", "fAlpha", "fX", "fY", "fZ"], library="pd")
        mask = df_trackextr.eval(cut_expression)        # create boolean mask
        # Apply the SAME filter to df_trackextr and df_track to keep them aligned (and keep only useful columns):
        df_trackextr = df_trackextr.loc[mask, ["fPt", "fEta", "fCharge", "fDcaXY"] ].reset_index(drop=True)
        df_track = df_track.loc[mask,].reset_index(drop=True)
        # merging in a single dataframe
        df_trackextr["fIndexCollisions"] = df_track["fIndexCollisions"] 
        df_trackextr["fAlpha"] = df_track["fAlpha"]
        df_trackextr["fX"] = df_track["fX"]
        df_trackextr["fY"] = df_track["fY"]
        df_trackextr["fZ"] = df_track["fZ"]
    
        # we cut rows where the fIndexCollision is negative (for some reason)
        valid = df_track["fIndexCollisions"] >= 0
        df_track = df_track[valid].reset_index(drop=True)          
        df_trackextr = df_trackextr[valid].reset_index(drop=True)  
    
        
        # Now we for correct fPosZ (and add that column)
        df_trackextr["fPosZ"] = df_coll.iloc[df_trackextr["fIndexCollisions"].values]["fPosZ"].values
      
        df_trackextr["fPosX"] = df_coll.iloc[df_trackextr["fIndexCollisions"].values]["fPosX"].values
    
        df_trackextr["fPosY"] = df_coll.iloc[df_trackextr["fIndexCollisions"].values]["fPosY"].values
    
        df_trackextr = df_trackextr[(df_trackextr["fPosZ"] < 10) & (df_trackextr["fPosZ"] > -10)].reset_index(drop=True)
        df_trackextr["fIndexCollisions"] += collision_offset   # Fix local fIndexCollisions → global index 
    
        # save results:
        # ALTERNATIVE 1: for the first cycle, let's copy the first dataframe, then we concatenate the next ones
        # if  i==0: df = df_trackextr
        # else: df = pd.concat([df, df_trackextr], ignore_index=True)
        # # ALTERNATIVE 2:
        list_of_df.append( df_trackextr )                  # add the dataframe in a list (we will concat them later)
        
        # Update offset for next loop
        collision_offset += len(df_coll)     
    
        # let's free the memory RAM of unused dataframes:
        del df_trackextr
        del df_track
        gc.collect()
    
    
    # # UNCOMMENT FOR ALTERNATIVE 2:
    df = pd.concat(list_of_df, ignore_index=True)
    
    # Merge everything in the total dataframe
    N = len(df)

    # moment columns:
    df["px"] = df["fPt"] * np.cos(df["fAlpha"])
    df["py"] = df["fPt"] * np.sin(df["fAlpha"])
    df["pz"] = df["fPt"] * np.sinh(df["fEta"])
    
    # energy column (differentiating pions and kaons):
    mass = np.where(df["fCharge"] > 0, m_pi, m_K)
    df["Ene"] = np.sqrt((df["fPt"] * np.cosh(df["fEta"]))**2 + mass**2)
    
    # Debug
    print("The starting dataframe has", len(df), "rows and ", len(df.columns), "columns.")
    memory = df.memory_usage(deep=True).sum() / (1024 ** 2)
    print(f"The starting dataframe occupies {memory:.2f} MB")
    # df.head()

    # ALTERNATIVE 3:
    # let's initialize some lists, then we will create a dataframe
    collision_indices = []
    track1_indices = []
    track2_indices = []
    dcaXY_products = []
    inv_masses = []
    inv_masses_approx = []
    pt_totals = []
    pz_totals = []
    SV_X = []
    SV_Y = []
    SV_Z = []
    decay_lengths = []
    cos_pointings = []
    
    counting=0 # debug variable
    
    # let's divide the dataframe for positive and negative charged
    df_pos = df[ df['fCharge']>0 ]
    df_neg = df[ df['fCharge']<0 ]
    
    # Iterate over each collision group
    for collision_idx in (df_neg['fIndexCollisions'].unique()):
        group_pos = df_pos[ df_pos['fIndexCollisions'] == collision_idx ]
        group_neg = df_neg[ df_neg['fIndexCollisions'] == collision_idx ]
    
        # Only collisions with at least a pair
        if len(group_pos) < 1:   continue
    
        # Reset index of the group to 0..N-1 and move original index in new column 'orig_index'
        group_pos = group_pos.reset_index().rename(columns={'index': 'orig_index'})
        group_neg = group_neg.reset_index().rename(columns={'index': 'orig_index'})
    
        # let's crate indexes for all possible pairs:
        combinat = itertools.product( range(len(group_neg)), range(len(group_pos)) )
    
        # Iterate over all unique pairs of tracks
        for combo in combinat:
            row_neg = group_neg.iloc[combo[0]]
            row_pos = group_pos.iloc[combo[1]]
    
            product_dcaXY = row_neg['fDcaXY'] * row_pos['fDcaXY']
            
            # INVARIANT MASS calculation
            pt1, pt2 = row_neg['fPt'], row_pos['fPt']
            # eta1, eta2 = row_neg['fEta'], row_pos['fEta']
            # phi1, phi2 = row_neg['fAlpha'], row_pos['fAlpha']
            # delta_eta = eta1 - eta2
            # delta_phi = phi1 - phi2
    
            # approximation formula
            # inv_mass_approx = np.sqrt(2 * pt1 * pt2 * (np.cosh(delta_eta) - np.cos(delta_phi)))
    
            # exact formula:
            E1 = row_neg['Ene']
            E2 = row_pos['Ene']
            px1 = row_neg["px"]
            py1 = row_neg["py"]
            pz1 = row_neg["pz"]
            px2 = row_pos["px"]
            py2 = row_pos["py"]
            pz2 = row_pos["pz"]
            inv_mass = np.sqrt( (E1+E2)**2 - (px1+px2)**2 - (py1+py2)**2 - (pz1+pz2)**2 )
    
    
            # total transverse momentum of the D0 candidate (used later for sliced plots)
            pt_total = np.sqrt((px1 + px2)**2 + (py1 + py2)**2)
    
            # secondary vertex
            SV_coords = np.array( secondary_vertex(row_neg, row_pos) )
            # SV_X.append(SV_coords[0])
            # SV_Y.append(SV_coords[1])
            # SV_Z.append(SV_coords[2])
    
            # decay length: distance between PV and SV
            PV_coords = np.array( [row_pos["fPosX"], row_pos["fPosY"], row_pos["fPosZ"]] )
            decay_lengths.append( np.linalg.norm(SV_coords - PV_coords ) )
    
            # cosine of pointing angle: the latter is the angle between the direction of the mother particle and the line connecting PV and SV
            mother_direction = [px1+px2, py1+py2, pz1+pz2]
            flight_line = SV_coords - PV_coords
            cos_pointings.append ( np.dot(mother_direction, flight_line)/(np.linalg.norm(mother_direction)*np.linalg.norm(flight_line)) )
            
            # let's add the found pairs to the lists
            collision_indices.append(int(row_neg['fIndexCollisions']))
            # track1_indices.append(int(row_neg['orig_index']))
            # track2_indices.append(int(row_pos['orig_index']))
            dcaXY_products.append(product_dcaXY)
            inv_masses.append(inv_mass)
            # inv_masses_approx.append(inv_mass_approx)
            pt_totals.append(pt_total)
            pz_totals.append(pz1+pz2)
    
        # # let's free the memory RAM of unused dataframes:
        # # PROBLEM: THIS IS VERY SLOW!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
        # del group
        # gc.collect()
    
        # debug:
        counting += 1
        if counting % 50000 ==0: print(counting, end=' ')
    
    
    # create a dataframe with the result:
    df_pairs = pd.DataFrame({
        'collision_index': collision_indices,
        # 'track1_index': track1_indices,
        # 'track2_index': track2_indices,
        'dcaXY_product': dcaXY_products,
        'inv_mass': inv_masses,
        # 'inv_mass_approx': inv_masses_approx,
        'pt': pt_totals,
        'pz': pz_totals,
        # 'X_SV': SV_X,
        # 'Y_SV': SV_Y,
        # 'Z_SV': SV_Z,
        'decay_length': decay_lengths,
        'cos_pointing': cos_pointings
    })

    # Debug
    print("The final dataframe has", len(df_pairs), "rows")
    memory = df_pairs.memory_usage(deep=True).sum() / (1024 ** 2)
    print(f"The dataframe occupy {memory:.2f} MB")
    display( pd.concat([df_pairs.head(2),df_pairs.tail(3)]) )

    save_name = "pairs_" + "_".join([which_chunk, which_number, str(J)])
    df_pairs.to_pickle(save_name + ".pkl")

    # free memory from the dataframes used for this subset
    del df
    del df_pairs
    del collision_indices 
    del dcaXY_products
    del inv_masses
    del pt_totals
    del pz_totals
    del decay_lengths
    del cos_pointings
    gc.collect()

    print(f"Iteration {J+1} out of {N_SPLITS} done!")
    print(f"{save_name} created.")

The starting dataframe has 1035777 rows and  16 columns.
The starting dataframe occupies 67.17 MB
50000 100000 The final dataframe has 1473850 rows
The dataframe occupy 78.71 MB


,collision_index,dcaXY_product,inv_mass,pt,pz,decay_length,cos_pointing
0,1,-3.920097e-06,1.466251,2.550995,-0.295362,0.019764,-0.479430
1,1,1.705092e-06,2.598888,0.476026,-0.071934,0.028125,-0.252065
1473847,328530,-9.394554e-07,0.784535,1.169593,-0.026994,0.011926,0.782126
1473848,328530,-3.067913e-06,1.186273,0.798706,0.248661,0.013467,-0.808711
1473849,328530,8.916265e-07,0.848528,1.275424,-0.125633,0.072691,0.097662


Iteration 1 out of 10 done!
pairs_Chunk1300_001_006_0 created.
The starting dataframe has 1001572 rows and  16 columns.
The starting dataframe occupies 64.95 MB
50000 100000 The final dataframe has 1425572 rows
The dataframe occupy 76.13 MB


,collision_index,dcaXY_product,inv_mass,pt,pz,decay_length,cos_pointing
0,328535,3.202240e-07,1.298124,2.857873,0.972263,0.007777,-0.198862
1,328535,-2.102688e-07,0.926050,1.208677,-0.167351,0.029844,-0.181474
1425569,646196,3.120991e-06,2.159951,1.086269,-1.467719,0.037969,0.994115
1425570,646196,-6.706499e-06,1.625211,1.438804,-1.553639,0.028236,0.792522
1425571,646196,-4.220028e-06,1.627056,1.413372,-1.588313,0.024405,0.770932


Iteration 2 out of 10 done!
pairs_Chunk1300_001_006_1 created.
The starting dataframe has 999742 rows and  16 columns.
The starting dataframe occupies 64.83 MB
50000 100000 The final dataframe has 1425436 rows
The dataframe occupy 76.13 MB


...

Iteration 8 out of 10 done!
pairs_Chunk1300_001_006_7 created.
The starting dataframe has 750911 rows and  16 columns.
The starting dataframe occupies 48.70 MB
50000 The final dataframe has 1070501 rows
The dataframe occupy 57.17 MB


,collision_index,dcaXY_product,inv_mass,pt,pz,decay_length,cos_pointing
0,2473934,-6.300046e-07,0.995451,2.207042,0.791055,0.021610,0.103111
1,2473934,8.183280e-06,1.341907,1.486902,-0.094683,0.009382,0.634560
1070498,2713584,-3.243990e-06,1.280039,0.268215,0.716599,0.086693,0.381622
1070499,2713584,-1.240443e-06,2.071605,1.894974,-0.400610,0.038173,0.169661
1070500,2713584,-5.791017e-07,1.245881,0.368911,0.446624,0.003446,-0.849451


Iteration 9 out of 10 done!
pairs_Chunk1300_001_006_8 created.
The starting dataframe has 983561 rows and  16 columns.
The starting dataframe occupies 63.78 MB
50000 100000 The final dataframe has 1408426 rows
The dataframe occupy 75.22 MB


,collision_index,dcaXY_product,inv_mass,pt,pz,decay_length,cos_pointing
0,2713587,-0.000008,1.454900,0.930696,-0.042458,0.003816,0.931455
1,2713587,0.000017,0.783386,1.450209,0.132077,0.017267,-0.918939
1408423,3023779,-0.000005,2.252738,1.132577,0.459364,0.002781,-0.320721
1408424,3023779,0.000032,1.013496,1.343605,-0.249762,0.026317,0.506724
1408425,3023782,0.000010,1.316963,0.452792,-0.410209,0.018628,-0.561987


Iteration 10 out of 10 done!
pairs_Chunk1300_001_006_9 created.
